# ODRL Assertion Templates v2 — with GroupedStatement

Rewrites the ODRL policy and grant assertion templates using `nt:GroupedStatement`
so that Nanodash can properly render nested ODRL structures:

- **Permission** → action + target + optional constraint (purpose)
- **Prohibition** → action + target
- **Duty** → action + attributedParty

Supersedes the old flat templates:
- Policy: `https://w3id.org/np/RAfmYWze87u4ofAm6xhGfobp9KX_aTbItFrVfxhtQ8m0c`
- Grant: `https://w3id.org/np/RAd1X8liGs-fjmbkrN514sL3CueyOVOjX3Bax63br6HYM`

---

## Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from rdflib import Dataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF

NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
ORCID = Namespace("https://orcid.org/")
ODRL = Namespace("http://www.w3.org/ns/odrl/2/")
DPV = Namespace("https://w3id.org/dpv#")
FAIR = Namespace("https://fair2adapt.eu/ns/")

AUTHOR_ORCID = "0000-0002-1784-2920"
AUTHOR_NAME = "Anne Fouilloux"

# Broken v3 policy template to supersede (grant template v3 works fine)
OLD_POLICY_TEMPLATE = URIRef("https://w3id.org/np/RAVqycKi1xUPx3yWHiv4Yj1YV0YYpw6js-7exHiAnaJ70")
OLD_GRANT_TEMPLATE = URIRef("https://w3id.org/np/RANSf-wdE61UVowtVZDKt1RvEftLdTWdI7UfJb1qxLzXM")

OUTPUT_DIR = "output/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Setup complete")

---
## Policy Template (GroupedStatement — flat, no nesting)

Nanodash does NOT support nested GroupedStatements ([issue #153](https://github.com/knowledgepixels/nanodash/issues/153)).
So the permission + constraint triples are all in one flat GroupedStatement.

Produces this ODRL structure when filled in via Nanodash:

```turtle
<policy> a odrl:Offer ;
    odrl:target <dataset> ;
    odrl:permission _:perm .
_:perm odrl:action odrl:use ;
    odrl:constraint _:c .
_:c odrl:leftOperand odrl:purpose ;
    odrl:operator odrl:eq ;
    odrl:rightOperand dpv:AcademicResearch .
<policy> odrl:prohibition _:p .
_:p odrl:action odrl:distribute .
<policy> odrl:duty _:d .
_:d odrl:action odrl:attribute ;
    odrl:attributedParty <party> .
```

In [ ]:
def create_odrl_policy_template():
    """
    ODRL Access Policy template using nt:GroupedStatement.
    No nesting — Nanodash only supports one level of grouping.
    Permission + constraint triples are in one flat group.
    """
    TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")

    this_np = URIRef("http://purl.org/nanopub/temp/np/")
    head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
    assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
    provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
    pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")

    author_uri = ORCID[AUTHOR_ORCID]

    ds = Dataset()
    for prefix, ns in [("this", "http://purl.org/nanopub/temp/np/"),
                        ("sub", TEMP_NP), ("np", NP), ("dct", DCT),
                        ("nt", NT), ("npx", NPX), ("xsd", XSD),
                        ("rdfs", RDFS), ("orcid", ORCID), ("prov", PROV),
                        ("foaf", FOAF), ("rdf", RDF), ("odrl", ODRL),
                        ("dpv", DPV)]:
        ds.bind(prefix, ns)

    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))

    # ASSERTION
    a = ds.graph(assertion_graph)

    # Predicate labels for Nanodash
    a.add((RDF.type, RDFS.label, Literal("is a")))
    a.add((ODRL.target, RDFS.label, Literal("applies to dataset")))
    a.add((ODRL.action, RDFS.label, Literal("action")))
    a.add((ODRL.permission, RDFS.label, Literal("has permission")))
    a.add((ODRL.prohibition, RDFS.label, Literal("has prohibition")))
    a.add((ODRL.duty, RDFS.label, Literal("has duty")))
    a.add((ODRL.constraint, RDFS.label, Literal("with constraint")))
    a.add((ODRL.leftOperand, RDFS.label, Literal("constraint on")))
    a.add((ODRL.operator, RDFS.label, Literal("operator")))
    a.add((ODRL.rightOperand, RDFS.label, Literal("must be")))
    a.add((ODRL.attributedParty, RDFS.label, Literal("attributed to")))

    # Template metadata
    a.add((TEMP_NP.assertion, RDF.type, NT.AssertionTemplate))
    a.add((TEMP_NP.assertion, RDFS.label, Literal("ODRL Access Policy for FAIR Data")))
    a.add((TEMP_NP.assertion, DCT.description, Literal(
        "Defines an ODRL policy for controlling access to a FAIR dataset. "
        "Specifies permitted actions (with purpose constraints), "
        "prohibited actions, and attribution duties. "
        "Use this template to publish machine-readable access conditions "
        "for private research data."
    )))
    a.add((TEMP_NP.assertion, NT.hasTag, Literal("FAIR2Adapt")))
    a.add((TEMP_NP.assertion, NT.hasTag, Literal("ODRL")))
    a.add((TEMP_NP.assertion, NT.hasTag, Literal("Access Control")))
    a.add((TEMP_NP.assertion, NT.hasNanopubLabelPattern,
           Literal("ODRL policy: ${datasetUri}")))
    a.add((TEMP_NP.assertion, NT.hasTargetNanopubType, ODRL.Policy))

    # ── Top-level statements ──
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.st_type))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.st_target))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.permGroup))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.prohibGroup))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.dutyGroup))

    # st_type: policy rdf:type policyType
    a.add((TEMP_NP.st_type, RDF.subject, TEMP_NP.policyUri))
    a.add((TEMP_NP.st_type, RDF.predicate, RDF.type))
    a.add((TEMP_NP.st_type, RDF.object, TEMP_NP.policyType))

    # st_target: policy odrl:target dataset
    a.add((TEMP_NP.st_target, RDF.subject, TEMP_NP.policyUri))
    a.add((TEMP_NP.st_target, RDF.predicate, ODRL.target))
    a.add((TEMP_NP.st_target, RDF.object, TEMP_NP.datasetUri))

    # ══════════════════════════════════════════════
    # permGroup: FLAT group with permission + constraint triples
    #   policy odrl:permission _:perm .
    #   _:perm odrl:action <action> .
    #   _:perm odrl:constraint _:const .
    #   _:const odrl:leftOperand odrl:purpose .
    #   _:const odrl:operator odrl:eq .
    #   _:const odrl:rightOperand <purpose> .
    # ══════════════════════════════════════════════
    a.add((TEMP_NP.permGroup, RDF.type, NT.GroupedStatement))
    a.add((TEMP_NP.permGroup, RDF.type, NT.RepeatableStatement))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.perm_link))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.perm_action))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.const_link))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.const_leftOp))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.const_op))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.const_rightOp))

    # LocalResources for blank nodes
    a.add((TEMP_NP.permNode, RDF.type, NT.LocalResource))
    a.add((TEMP_NP.constNode, RDF.type, NT.LocalResource))

    # perm_link: policy odrl:permission _:permNode
    a.add((TEMP_NP.perm_link, RDF.subject, TEMP_NP.policyUri))
    a.add((TEMP_NP.perm_link, RDF.predicate, ODRL.permission))
    a.add((TEMP_NP.perm_link, RDF.object, TEMP_NP.permNode))

    # perm_action: _:permNode odrl:action <action>
    a.add((TEMP_NP.perm_action, RDF.subject, TEMP_NP.permNode))
    a.add((TEMP_NP.perm_action, RDF.predicate, ODRL.action))
    a.add((TEMP_NP.perm_action, RDF.object, TEMP_NP.permittedAction))

    # const_link: _:permNode odrl:constraint _:constNode
    a.add((TEMP_NP.const_link, RDF.subject, TEMP_NP.permNode))
    a.add((TEMP_NP.const_link, RDF.predicate, ODRL.constraint))
    a.add((TEMP_NP.const_link, RDF.object, TEMP_NP.constNode))

    # const_leftOp: _:constNode odrl:leftOperand odrl:purpose (fixed)
    a.add((TEMP_NP.const_leftOp, RDF.subject, TEMP_NP.constNode))
    a.add((TEMP_NP.const_leftOp, RDF.predicate, ODRL.leftOperand))
    a.add((TEMP_NP.const_leftOp, RDF.object, ODRL.purpose))
    a.add((ODRL.purpose, RDFS.label, Literal("Purpose")))

    # const_op: _:constNode odrl:operator odrl:eq (fixed)
    a.add((TEMP_NP.const_op, RDF.subject, TEMP_NP.constNode))
    a.add((TEMP_NP.const_op, RDF.predicate, ODRL.operator))
    a.add((TEMP_NP.const_op, RDF.object, ODRL.eq))
    a.add((ODRL.eq, RDFS.label, Literal("equals")))

    # const_rightOp: _:constNode odrl:rightOperand <purpose> (placeholder)
    a.add((TEMP_NP.const_rightOp, RDF.subject, TEMP_NP.constNode))
    a.add((TEMP_NP.const_rightOp, RDF.predicate, ODRL.rightOperand))
    a.add((TEMP_NP.const_rightOp, RDF.object, TEMP_NP.purposeConstraint))

    # ══════════════════════════════════════════════
    # prohibGroup: policy odrl:prohibition [ odrl:action X ]
    # ══════════════════════════════════════════════
    a.add((TEMP_NP.prohibGroup, RDF.type, NT.GroupedStatement))
    a.add((TEMP_NP.prohibGroup, RDF.type, NT.OptionalStatement))
    a.add((TEMP_NP.prohibGroup, RDF.type, NT.RepeatableStatement))
    a.add((TEMP_NP.prohibGroup, NT.hasStatement, TEMP_NP.prohib_link))
    a.add((TEMP_NP.prohibGroup, NT.hasStatement, TEMP_NP.prohib_action))

    a.add((TEMP_NP.prohibNode, RDF.type, NT.LocalResource))

    a.add((TEMP_NP.prohib_link, RDF.subject, TEMP_NP.policyUri))
    a.add((TEMP_NP.prohib_link, RDF.predicate, ODRL.prohibition))
    a.add((TEMP_NP.prohib_link, RDF.object, TEMP_NP.prohibNode))

    a.add((TEMP_NP.prohib_action, RDF.subject, TEMP_NP.prohibNode))
    a.add((TEMP_NP.prohib_action, RDF.predicate, ODRL.action))
    a.add((TEMP_NP.prohib_action, RDF.object, TEMP_NP.prohibitedAction))

    # ══════════════════════════════════════════════
    # dutyGroup: policy odrl:duty [ odrl:action X ; odrl:attributedParty Y ]
    # ══════════════════════════════════════════════
    a.add((TEMP_NP.dutyGroup, RDF.type, NT.GroupedStatement))
    a.add((TEMP_NP.dutyGroup, RDF.type, NT.OptionalStatement))
    a.add((TEMP_NP.dutyGroup, NT.hasStatement, TEMP_NP.duty_link))
    a.add((TEMP_NP.dutyGroup, NT.hasStatement, TEMP_NP.duty_action))
    a.add((TEMP_NP.dutyGroup, NT.hasStatement, TEMP_NP.duty_party))

    a.add((TEMP_NP.dutyNode, RDF.type, NT.LocalResource))

    a.add((TEMP_NP.duty_link, RDF.subject, TEMP_NP.policyUri))
    a.add((TEMP_NP.duty_link, RDF.predicate, ODRL.duty))
    a.add((TEMP_NP.duty_link, RDF.object, TEMP_NP.dutyNode))

    a.add((TEMP_NP.duty_action, RDF.subject, TEMP_NP.dutyNode))
    a.add((TEMP_NP.duty_action, RDF.predicate, ODRL.action))
    a.add((TEMP_NP.duty_action, RDF.object, TEMP_NP.dutyAction))

    a.add((TEMP_NP.duty_party, RDF.type, NT.OptionalStatement))
    a.add((TEMP_NP.duty_party, RDF.subject, TEMP_NP.dutyNode))
    a.add((TEMP_NP.duty_party, RDF.predicate, ODRL.attributedParty))
    a.add((TEMP_NP.duty_party, RDF.object, TEMP_NP.attributionParty))

    # ══════════════════════════════════════════════
    # PLACEHOLDERS
    # ══════════════════════════════════════════════

    # policyUri
    a.add((TEMP_NP.policyUri, RDF.type, NT.UriPlaceholder))
    a.add((TEMP_NP.policyUri, RDF.type, NT.IntroducedResource))
    a.add((TEMP_NP.policyUri, RDFS.label, Literal("URI of the policy")))
    a.add((TEMP_NP.policyUri, NT.hasPrefix, Literal("https://fair2adapt.eu/policy/")))
    a.add((TEMP_NP.policyUri, NT.hasPrefixLabel, Literal("fair2adapt-policy")))

    # policyType (restricted choice)
    a.add((TEMP_NP.policyType, RDF.type, NT.RestrictedChoicePlaceholder))
    a.add((TEMP_NP.policyType, RDFS.label, Literal("Type of ODRL policy")))
    a.add((TEMP_NP.policyType, NT.possibleValue, ODRL.Offer))
    a.add((TEMP_NP.policyType, NT.possibleValue, ODRL.Set))
    a.add((ODRL.Offer, RDFS.label, Literal("Offer - data available under conditions")))
    a.add((ODRL.Set, RDFS.label, Literal("Set - general policy statement")))

    # datasetUri
    a.add((TEMP_NP.datasetUri, RDF.type, NT.UriPlaceholder))
    a.add((TEMP_NP.datasetUri, RDFS.label, Literal("URI of the dataset this policy applies to")))
    a.add((TEMP_NP.datasetUri, NT.hasPrefix, Literal("https://fair2adapt.eu/data/")))
    a.add((TEMP_NP.datasetUri, NT.hasPrefixLabel, Literal("fair2adapt-data")))

    # permittedAction (restricted choice)
    a.add((TEMP_NP.permittedAction, RDF.type, NT.RestrictedChoicePlaceholder))
    a.add((TEMP_NP.permittedAction, RDFS.label, Literal("Permitted action")))
    for action_uri, label in [
        (ODRL.use, "Use"), (ODRL.reproduce, "Reproduce"),
        (ODRL.distribute, "Distribute"), (ODRL.derive, "Derive"),
        (ODRL.present, "Present"), (ODRL.modify, "Modify"),
    ]:
        a.add((TEMP_NP.permittedAction, NT.possibleValue, action_uri))
        a.add((action_uri, RDFS.label, Literal(label)))

    # purposeConstraint (restricted choice, DPV)
    a.add((TEMP_NP.purposeConstraint, RDF.type, NT.RestrictedChoicePlaceholder))
    a.add((TEMP_NP.purposeConstraint, RDFS.label, Literal("Required purpose for access")))
    for purpose_uri, label in [
        (DPV.AcademicResearch, "Academic Research"),
        (DPV.ScientificResearch, "Scientific Research"),
        (DPV.NonCommercialResearch, "Non-Commercial Research"),
        (DPV.PublicBenefit, "Public Benefit"),
    ]:
        a.add((TEMP_NP.purposeConstraint, NT.possibleValue, purpose_uri))
        a.add((purpose_uri, RDFS.label, Literal(label)))

    # prohibitedAction (restricted choice)
    a.add((TEMP_NP.prohibitedAction, RDF.type, NT.RestrictedChoicePlaceholder))
    a.add((TEMP_NP.prohibitedAction, RDFS.label, Literal("Prohibited action")))
    for action_uri, label in [
        (ODRL.distribute, "Distribute"), (ODRL.commercialize, "Commercialize"),
        (ODRL.sell, "Sell"), (ODRL.modify, "Modify"),
    ]:
        a.add((TEMP_NP.prohibitedAction, NT.possibleValue, action_uri))
        a.add((action_uri, RDFS.label, Literal(label)))

    # dutyAction (restricted choice)
    a.add((TEMP_NP.dutyAction, RDF.type, NT.RestrictedChoicePlaceholder))
    a.add((TEMP_NP.dutyAction, RDFS.label, Literal("Required duty action")))
    a.add((TEMP_NP.dutyAction, NT.possibleValue, ODRL.attribute))
    a.add((TEMP_NP.dutyAction, NT.possibleValue, ODRL.compensate))
    a.add((TEMP_NP.dutyAction, NT.possibleValue, ODRL.obtainConsent))
    a.add((ODRL.attribute, RDFS.label, Literal("Attribute")))
    a.add((ODRL.compensate, RDFS.label, Literal("Compensate")))
    a.add((ODRL.obtainConsent, RDFS.label, Literal("Obtain Consent")))

    # attributionParty
    a.add((TEMP_NP.attributionParty, RDF.type, NT.UriPlaceholder))
    a.add((TEMP_NP.attributionParty, RDFS.label,
           Literal("URI of party to attribute (e.g. https://fair2adapt-eosc.eu)")))

    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))

    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author_uri, FOAF.name, Literal(AUTHOR_NAME)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef("https://fair2adapt-eosc.eu")))
    pubinfo.add((this_np, RDFS.label, Literal("Assertion template: ODRL Access Policy for FAIR Data")))
    pubinfo.add((this_np, NPX.hasNanopubType, NT.AssertionTemplate))
    pubinfo.add((this_np, NPX.supersedes, OLD_POLICY_TEMPLATE))

    return ds

ds_policy = create_odrl_policy_template()
policy_template_file = Path(OUTPUT_DIR) / "odrl_policy_template_v2.trig"
ds_policy.serialize(destination=str(policy_template_file), format='trig')
print(f"Generated: {policy_template_file}")

In [10]:
print(f"Preview of {policy_template_file}:\n")
print("=" * 80)
with open(policy_template_file, 'r') as f:
    print(f.read())

Preview of output/odrl_policy_template_v2.trig:

@prefix dct: <http://purl.org/dc/terms/> .
@prefix dpv: <https://w3id.org/dpv#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix odrl: <http://www.w3.org/ns/odrl/2/> .
@prefix orcid: <https://orcid.org/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sub: <http://purl.org/nanopub/temp/np/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

sub:Head {
    sub: a np:Nanopublication ;
        np:hasAssertion sub:assertion ;
        np:hasProvenance sub:provenance ;
        np:hasPublicationInfo sub:pubinfo .
}

sub:provenance {
    sub:assertion prov:wasAttributedTo orcid:0000-0002-1784-2920 .
}

sub:pubinfo {
    sub: rdfs:label "Assertion template: ODRL Access Policy f

---
## Grant Template (GroupedStatement)

Produces this ODRL structure when filled in via Nanodash:

```turtle
<grant> a odrl:Agreement ;
    odrl:target <dataset> ;
    odrl:assignee <did> ;
    odrl:permission [
        odrl:action odrl:use
    ] ;
    fair:underPolicy <policy-nanopub> ;
    prov:generatedAtTime "2026-..."^^xsd:dateTime .
```

In [11]:
def create_odrl_grant_template():
    """
    ODRL Access Grant template using nt:GroupedStatement for permission.
    Supersedes the old flat template.
    """
    TEMP_NP = Namespace("http://purl.org/nanopub/temp/np/")

    this_np = URIRef("http://purl.org/nanopub/temp/np/")
    head_graph = URIRef("http://purl.org/nanopub/temp/np/Head")
    assertion_graph = URIRef("http://purl.org/nanopub/temp/np/assertion")
    provenance_graph = URIRef("http://purl.org/nanopub/temp/np/provenance")
    pubinfo_graph = URIRef("http://purl.org/nanopub/temp/np/pubinfo")

    author_uri = ORCID[AUTHOR_ORCID]

    ds = Dataset()
    for prefix, ns in [("this", "http://purl.org/nanopub/temp/np/"),
                        ("sub", TEMP_NP), ("np", NP), ("dct", DCT),
                        ("nt", NT), ("npx", NPX), ("xsd", XSD),
                        ("rdfs", RDFS), ("orcid", ORCID), ("prov", PROV),
                        ("foaf", FOAF), ("rdf", RDF), ("odrl", ODRL),
                        ("fair", FAIR)]:
        ds.bind(prefix, ns)

    # HEAD
    head = ds.graph(head_graph)
    head.add((this_np, RDF.type, NP.Nanopublication))
    head.add((this_np, NP.hasAssertion, assertion_graph))
    head.add((this_np, NP.hasProvenance, provenance_graph))
    head.add((this_np, NP.hasPublicationInfo, pubinfo_graph))

    # ASSERTION
    a = ds.graph(assertion_graph)

    # Predicate labels
    a.add((RDF.type, RDFS.label, Literal("is a")))
    a.add((ODRL.target, RDFS.label, Literal("grants access to dataset")))
    a.add((ODRL.assignee, RDFS.label, Literal("is granted to")))
    a.add((ODRL.action, RDFS.label, Literal("for action")))
    a.add((ODRL.permission, RDFS.label, Literal("has permission")))
    a.add((FAIR.underPolicy, RDFS.label, Literal("under ODRL policy")))
    a.add((PROV.generatedAtTime, RDFS.label, Literal("granted at time")))

    # Template metadata
    a.add((TEMP_NP.assertion, RDF.type, NT.AssertionTemplate))
    a.add((TEMP_NP.assertion, RDFS.label, Literal("ODRL Access Grant for FAIR Data")))
    a.add((TEMP_NP.assertion, DCT.description, Literal(
        "Records that a specific DID was granted access to a dataset "
        "under an ODRL policy. Serves as an immutable audit trail "
        "for data access in the FAIR2Adapt project."
    )))
    a.add((TEMP_NP.assertion, NT.hasTag, Literal("FAIR2Adapt")))
    a.add((TEMP_NP.assertion, NT.hasTag, Literal("ODRL")))
    a.add((TEMP_NP.assertion, NT.hasTag, Literal("Access Grant")))
    a.add((TEMP_NP.assertion, NT.hasNanopubLabelPattern,
           Literal("Access grant: ${datasetUri} -> ${assigneeDid}")))
    a.add((TEMP_NP.assertion, NT.hasTargetNanopubType, ODRL.Agreement))

    # ── Top-level statements ──
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.st_type))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.st_target))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.st_assignee))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.permGroup))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.st_policy))
    a.add((TEMP_NP.assertion, NT.hasStatement, TEMP_NP.st_time))

    # st_type: grant rdf:type odrl:Agreement (fixed)
    a.add((TEMP_NP.st_type, RDF.subject, TEMP_NP.grantUri))
    a.add((TEMP_NP.st_type, RDF.predicate, RDF.type))
    a.add((TEMP_NP.st_type, RDF.object, ODRL.Agreement))

    # st_target: grant odrl:target dataset
    a.add((TEMP_NP.st_target, RDF.subject, TEMP_NP.grantUri))
    a.add((TEMP_NP.st_target, RDF.predicate, ODRL.target))
    a.add((TEMP_NP.st_target, RDF.object, TEMP_NP.datasetUri))

    # st_assignee: grant odrl:assignee did
    a.add((TEMP_NP.st_assignee, RDF.subject, TEMP_NP.grantUri))
    a.add((TEMP_NP.st_assignee, RDF.predicate, ODRL.assignee))
    a.add((TEMP_NP.st_assignee, RDF.object, TEMP_NP.assigneeDid))

    # ── permGroup: grant odrl:permission [ odrl:action X ] (repeatable) ──
    a.add((TEMP_NP.permGroup, RDF.type, NT.GroupedStatement))
    a.add((TEMP_NP.permGroup, RDF.type, NT.RepeatableStatement))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.grant_perm_link))
    a.add((TEMP_NP.permGroup, NT.hasStatement, TEMP_NP.grant_perm_action))

    # LocalResource for the permission blank node
    a.add((TEMP_NP.grantPermNode, RDF.type, NT.LocalResource))

    # grant_perm_link: grant odrl:permission _:grantPermNode
    a.add((TEMP_NP.grant_perm_link, RDF.subject, TEMP_NP.grantUri))
    a.add((TEMP_NP.grant_perm_link, RDF.predicate, ODRL.permission))
    a.add((TEMP_NP.grant_perm_link, RDF.object, TEMP_NP.grantPermNode))

    # grant_perm_action: _:grantPermNode odrl:action <action>
    a.add((TEMP_NP.grant_perm_action, RDF.subject, TEMP_NP.grantPermNode))
    a.add((TEMP_NP.grant_perm_action, RDF.predicate, ODRL.action))
    a.add((TEMP_NP.grant_perm_action, RDF.object, TEMP_NP.grantedAction))

    # st_policy: grant fair:underPolicy policyNanopub
    a.add((TEMP_NP.st_policy, RDF.subject, TEMP_NP.grantUri))
    a.add((TEMP_NP.st_policy, RDF.predicate, FAIR.underPolicy))
    a.add((TEMP_NP.st_policy, RDF.object, TEMP_NP.policyNanopubUri))

    # st_time: grant prov:generatedAtTime timestamp
    a.add((TEMP_NP.st_time, RDF.subject, TEMP_NP.grantUri))
    a.add((TEMP_NP.st_time, RDF.predicate, PROV.generatedAtTime))
    a.add((TEMP_NP.st_time, RDF.object, TEMP_NP.grantTimestamp))

    # ══════════════════════════════════════════════
    # PLACEHOLDERS
    # ══════════════════════════════════════════════

    # grantUri
    a.add((TEMP_NP.grantUri, RDF.type, NT.UriPlaceholder))
    a.add((TEMP_NP.grantUri, RDF.type, NT.IntroducedResource))
    a.add((TEMP_NP.grantUri, RDFS.label, Literal("Grant identifier")))

    # datasetUri
    a.add((TEMP_NP.datasetUri, RDF.type, NT.UriPlaceholder))
    a.add((TEMP_NP.datasetUri, RDFS.label, Literal("URI of the dataset")))
    a.add((TEMP_NP.datasetUri, NT.hasPrefix, Literal("https://fair2adapt.eu/data/")))
    a.add((TEMP_NP.datasetUri, NT.hasPrefixLabel, Literal("fair2adapt-data")))

    # assigneeDid
    a.add((TEMP_NP.assigneeDid, RDF.type, NT.UriPlaceholder))
    a.add((TEMP_NP.assigneeDid, RDFS.label,
           Literal("DID of the requester (e.g. did:web:researcher.example.org)")))

    # grantedAction (restricted choice)
    a.add((TEMP_NP.grantedAction, RDF.type, NT.RestrictedChoicePlaceholder))
    a.add((TEMP_NP.grantedAction, RDFS.label, Literal("Granted action")))
    a.add((TEMP_NP.grantedAction, NT.possibleValue, ODRL.use))
    a.add((TEMP_NP.grantedAction, NT.possibleValue, ODRL.reproduce))
    a.add((TEMP_NP.grantedAction, NT.possibleValue, ODRL.distribute))

    # policyNanopubUri
    a.add((TEMP_NP.policyNanopubUri, RDF.type, NT.UriPlaceholder))
    a.add((TEMP_NP.policyNanopubUri, RDFS.label,
           Literal("Nanopub URI of the ODRL policy this grant is under")))
    a.add((TEMP_NP.policyNanopubUri, NT.hasPrefix, Literal("https://w3id.org/np/")))
    a.add((TEMP_NP.policyNanopubUri, NT.hasPrefixLabel, Literal("nanopub")))

    # grantTimestamp (literal)
    a.add((TEMP_NP.grantTimestamp, RDF.type, NT.LiteralPlaceholder))
    a.add((TEMP_NP.grantTimestamp, RDFS.label, Literal("Timestamp of access grant")))
    a.add((TEMP_NP.grantTimestamp, NT.hasDatatype, XSD.dateTime))

    # PROVENANCE
    provenance = ds.graph(provenance_graph)
    provenance.add((assertion_graph, PROV.wasAttributedTo, author_uri))

    # PUBINFO
    pubinfo = ds.graph(pubinfo_graph)
    pubinfo.add((author_uri, FOAF.name, Literal(AUTHOR_NAME)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
    pubinfo.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pubinfo.add((this_np, DCT.creator, author_uri))
    pubinfo.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pubinfo.add((this_np, NPX.wasCreatedAt, URIRef("https://fair2adapt-eosc.eu")))
    pubinfo.add((this_np, RDFS.label, Literal("Assertion template: ODRL Access Grant for FAIR Data")))
    pubinfo.add((this_np, NPX.hasNanopubType, NT.AssertionTemplate))
    pubinfo.add((this_np, NPX.supersedes, OLD_GRANT_TEMPLATE))

    return ds

ds_grant = create_odrl_grant_template()
grant_template_file = Path(OUTPUT_DIR) / "odrl_grant_template_v2.trig"
ds_grant.serialize(destination=str(grant_template_file), format='trig')
print(f"Generated: {grant_template_file}")

Generated: output/odrl_grant_template_v2.trig


In [12]:
print(f"Preview of {grant_template_file}:\n")
print("=" * 80)
with open(grant_template_file, 'r') as f:
    print(f.read())

Preview of output/odrl_grant_template_v2.trig:

@prefix dct: <http://purl.org/dc/terms/> .
@prefix fair: <https://fair2adapt.eu/ns/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix odrl: <http://www.w3.org/ns/odrl/2/> .
@prefix orcid: <https://orcid.org/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sub: <http://purl.org/nanopub/temp/np/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

sub:Head {
    sub: a np:Nanopublication ;
        np:hasAssertion sub:assertion ;
        np:hasProvenance sub:provenance ;
        np:hasPublicationInfo sub:pubinfo .
}

sub:provenance {
    sub:assertion prov:wasAttributedTo orcid:0000-0002-1784-2920 .
}

sub:pubinfo {
    sub: rdfs:label "Assertion template: ODRL Access Gran

---
## Sign & Publish

Set `PUBLISH = True` when ready. The new templates will supersede the old ones.

In [13]:
PUBLISH = True  # Set to True when ready
USE_TEST_SERVER = False
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/ai-profile/profile.yml"
PROFILE_PATH = "/Users/annef/Documents/ScienceLive/annefou-profile/profile.yml"

In [ ]:
if PUBLISH:
    from nanopub import Nanopub, NanopubConf, load_profile

    profile = load_profile(PROFILE_PATH)
    print(f"Loaded profile: {profile.name}")

    conf = NanopubConf(profile=profile, use_test_server=USE_TEST_SERVER)

    # Only publish the policy template — grant template already works
    template_files = [policy_template_file]
    published = {}

    for trig_file in template_files:
        np_obj = Nanopub(rdf=trig_file, conf=conf)
        np_obj.sign()

        signed_path = trig_file.with_suffix('.signed.trig')
        np_obj.store(signed_path)
        print(f"Signed: {signed_path}")

        np_obj.publish()
        published[trig_file.stem] = np_obj.source_uri
        print(f"Published: {np_obj.source_uri}")

    print("\n" + "=" * 80)
    print("New template URIs:")
    print("=" * 80)
    print(f"  Policy template: {published.get('odrl_policy_template_v2', 'N/A')}")
    print(f"  Grant template (unchanged): https://w3id.org/np/RAeRMv6jOibLPIYBMOGu_FsX6NQ6B59KJCgCFkue4z7Ac")
else:
    print("Publishing disabled. Set PUBLISH = True when ready.")
    print(f"\nGenerated files:")
    print(f"  {policy_template_file}")
    print(f"  {grant_template_file}")

---
## Retract old flat templates

Retract the original v1 flat templates that don't work in Nanodash.